# Treinamento do Modelo de Estimativa de Peso de Gado

Este notebook treina um modelo MobileNetV3-Small via transfer learning para estimar o peso de bovinos a partir de fotos.

**Resultado:** arquivo `cattle_weight_model.tflite` para uso offline no app.

## Requisitos
- Google Colab com GPU (T4 recomendado)
- Conta Kaggle com API key configurada (para download do dataset)

In [ ]:
# 1. Instalar dependencias
!pip install -q tensorflow pandas matplotlib scikit-learn Pillow

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from PIL import Image
import glob
import json

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponivel: {tf.config.list_physical_devices('GPU')}")

## 2. Montar Google Drive e Carregar Dataset

O dataset deve estar descompactado em `Meu Drive/KaggleDatasets/dataset`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dataset_path = '/content/drive/MyDrive/KaggleDatasets/dataset'

assert os.path.exists(dataset_path), f"Pasta nao encontrada: {dataset_path}. Verifique se o dataset esta em 'Meu Drive/KaggleDatasets/dataset'"

print(f"Dataset encontrado em: {dataset_path}")

# Listar conteudo
for root, dirs, files_list in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for file in files_list[:10]:
            print(f'{subindent}{file}')
        if len(files_list) > 10:
            print(f'{subindent}... e mais {len(files_list) - 10} arquivos')

## 3. Carregar e Explorar os Dados

O dataset usa a convencao de nome de arquivo:
- B2/B3: `<id>_<side/rear>_<manual-weight>_<sex>.<ext>`
- B4: `<id>_<batchid>-<subbatchid>_<side/rear>_<manual-weight>_<sex>.<ext>`

Usamos as imagens **Side** da pasta `Pixel` (vista lateral, melhor para estimativa de peso).

In [ ]:
import re

# Pastas com imagens Side (vista lateral) dentro de Pixel
side_image_dirs = []

# B2: Pixel/B2/Side/images e Pixel/B2/Side_2/images
for sub in ['Side', 'Side_2']:
    d = os.path.join(dataset_path, 'Pixel', 'B2', sub, 'images')
    if os.path.exists(d):
        side_image_dirs.append(d)
        print(f"Encontrado: {d} ({len(os.listdir(d))} arquivos)")

# B3: Pixel/B3/images (apenas Side conforme README)
d = os.path.join(dataset_path, 'Pixel', 'B3', 'images')
if os.path.exists(d):
    side_image_dirs.append(d)
    print(f"Encontrado: {d} ({len(os.listdir(d))} arquivos)")

# B4: Pixel/B4/Side/images
d = os.path.join(dataset_path, 'Pixel', 'B4', 'Side', 'images')
if os.path.exists(d):
    side_image_dirs.append(d)
    print(f"Encontrado: {d} ({len(os.listdir(d))} arquivos)")

print(f"\nTotal de pastas Side encontradas: {len(side_image_dirs)}")

In [ ]:
def extract_weight_from_filename(filename):
    """
    Extrai peso do nome do arquivo.
    B2/B3: <id>_<side/rear>_<manual-weight>_<sex>.<ext>
    B4:    <id>_<batchid>-<subbatchid>_<side/rear>_<manual-weight>_<sex>.<ext>
    """
    name = os.path.splitext(filename)[0]  # remove extensao
    parts = name.split('_')
    
    # O peso e o penultimo campo (antes do sexo)
    # Tentar de tras para frente: ultimo = sexo, penultimo = peso
    if len(parts) >= 4:
        try:
            return float(parts[-2])
        except ValueError:
            pass
    
    # Fallback: procurar qualquer numero que pareca um peso (entre 100 e 1000)
    for part in reversed(parts):
        try:
            val = float(part)
            if 50 <= val <= 1500:
                return val
        except ValueError:
            continue
    
    return None

# Carregar imagens e pesos
image_paths = []
weights = []
skipped = 0

for img_dir in side_image_dirs:
    for filename in os.listdir(img_dir):
        if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        
        weight = extract_weight_from_filename(filename)
        if weight is not None:
            image_paths.append(os.path.join(img_dir, filename))
            weights.append(weight)
        else:
            skipped += 1

print(f"Total de amostras validas: {len(image_paths)}")
print(f"Arquivos ignorados (peso nao encontrado): {skipped}")

if weights:
    print(f"\nPeso minimo: {min(weights):.1f} kg")
    print(f"Peso maximo: {max(weights):.1f} kg")
    print(f"Peso medio: {np.mean(weights):.1f} kg")
    print(f"Desvio padrao: {np.std(weights):.1f} kg")

# Mostrar exemplos de parsing
print("\nExemplos de parsing:")
for img_dir in side_image_dirs[:1]:
    for f in os.listdir(img_dir)[:5]:
        w = extract_weight_from_filename(f)
        print(f"  {f} -> {w} kg")

In [ ]:
# Visualizar distribuicao de pesos e amostras
if weights:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histograma de pesos
    axes[0].hist(weights, bins=30, color='#3366FF', alpha=0.7, edgecolor='white')
    axes[0].set_xlabel('Peso (kg)')
    axes[0].set_ylabel('Frequencia')
    axes[0].set_title('Distribuicao de Pesos')
    axes[0].axvline(np.mean(weights), color='red', linestyle='--', label=f'Media: {np.mean(weights):.0f} kg')
    axes[0].legend()

    # Amostras aleatorias
    n_samples = min(6, len(image_paths))
    indices = np.random.choice(len(image_paths), n_samples, replace=False)
    axes[1].axis('off')
    axes[1].set_title('Amostras do Dataset')

    plt.tight_layout()
    plt.show()

    # Grid de amostras
    fig2, axes2 = plt.subplots(2, 3, figsize=(12, 8))
    for i, idx in enumerate(indices):
        ax = axes2[i // 3][i % 3]
        img = Image.open(image_paths[idx]).resize((224, 224))
        ax.imshow(img)
        ax.set_title(f'{weights[idx]:.0f} kg', fontsize=12)
        ax.axis('off')
    plt.suptitle('Amostras do Dataset', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Pre-processamento e Data Pipeline

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Split train/val/test (70/15/15)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    image_paths, weights, test_size=0.15, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.176, random_state=42  # 0.176 * 0.85 ~= 0.15
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# Normalizar pesos para melhor convergencia
weight_mean = np.mean(y_train)
weight_std = np.std(y_train)
print(f"Peso medio (train): {weight_mean:.1f} kg, std: {weight_std:.1f} kg")

def normalize_weight(w):
    return (w - weight_mean) / weight_std

def denormalize_weight(w):
    return w * weight_std + weight_mean

In [ ]:
def load_and_preprocess_image(path, weight):
    """Carrega imagem, resize para 224x224, normaliza pixels [0,1]"""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = img / 255.0
    return img, weight

def augment_image(img, weight):
    """Data augmentation para treino"""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, weight

def create_dataset(paths, weights_list, is_training=False):
    """Cria tf.data.Dataset"""
    # Normalizar pesos
    normalized_weights = [normalize_weight(w) for w in weights_list]
    
    ds = tf.data.Dataset.from_tensor_slices((paths, normalized_weights))
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    
    if is_training:
        ds = ds.shuffle(1000)
        ds = ds.map(augment_image, num_parallel_calls=AUTOTUNE)
    
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = create_dataset(X_train, y_train, is_training=True)
val_ds = create_dataset(X_val, y_val, is_training=False)
test_ds = create_dataset(X_test, y_test, is_training=False)

# Verificar um batch
for images, labels in train_ds.take(1):
    print(f"Batch shape: {images.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Pixel range: [{images.numpy().min():.2f}, {images.numpy().max():.2f}]")
    print(f"Weight range (normalized): [{labels.numpy().min():.2f}, {labels.numpy().max():.2f}]")

## 5. Construir o Modelo

MobileNetV3-Small com transfer learning do ImageNet.

In [ ]:
def build_model():
    base_model = tf.keras.applications.MobileNetV3Small(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    # Freeze base layers
    base_model.trainable = False
    
    model = keras.Sequential([
        base_model,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation='linear')  # Regressao
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='mse',
        metrics=['mae']
    )
    
    return model, base_model

model, base_model = build_model()
model.summary()

## 6. Treinamento - Fase 1 (Feature Extraction)

Treinar apenas as camadas novas (base congelada).

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6
    )
]

print("Fase 1: Feature Extraction (base congelada)")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

## 7. Fine-tuning

Descongelar as ultimas camadas da base e re-treinar com learning rate menor.

In [ ]:
# Descongelar ultimas camadas da base
base_model.trainable = True

# Congelar tudo exceto as ultimas 30 camadas
fine_tune_at = max(0, len(base_model.layers) - 30)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Camadas treinaveis na base: {trainable_count}/{len(base_model.layers)}")

# Recompilar com learning rate menor
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='mse',
    metrics=['mae']
)

callbacks_ft = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7
    )
]

print("\nFase 2: Fine-tuning")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks_ft
)

## 8. Avaliacao

In [ ]:
# Avaliar no conjunto de teste
test_loss, test_mae = model.evaluate(test_ds)
print(f"\n=== Resultado no Teste ===")
print(f"MSE (normalizado): {test_loss:.4f}")
print(f"MAE (normalizado): {test_mae:.4f}")
print(f"MAE (kg): {test_mae * weight_std:.1f} kg")
print(f"RMSE (kg): {np.sqrt(test_loss) * weight_std:.1f} kg")

# Predicoes vs Real
y_pred_norm = model.predict(test_ds).flatten()
y_true_norm = np.concatenate([y for _, y in test_ds], axis=0)

y_pred = denormalize_weight(y_pred_norm)
y_true = denormalize_weight(y_true_norm)

# Grafico Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_true, y_pred, alpha=0.6, s=20, color='#3366FF')
axes[0].plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], 'r--', linewidth=2)
axes[0].set_xlabel('Peso Real (kg)')
axes[0].set_ylabel('Peso Previsto (kg)')
axes[0].set_title('Previsto vs Real')
axes[0].grid(True, alpha=0.3)

# Distribuicao dos erros
errors = y_pred - y_true
axes[1].hist(errors, bins=30, color='#3366FF', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Erro (kg)')
axes[1].set_ylabel('Frequencia')
axes[1].set_title(f'Distribuicao dos Erros (MAE: {np.mean(np.abs(errors)):.1f} kg)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Erro percentual
pct_errors = np.abs(errors) / y_true * 100
print(f"\nErro percentual medio: {np.mean(pct_errors):.1f}%")
print(f"Erro percentual mediano: {np.median(pct_errors):.1f}%")
print(f"90% dos erros abaixo de: {np.percentile(pct_errors, 90):.1f}%")

In [ ]:
# Historico de treinamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Combinar historicos
all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']
all_mae = history1.history['mae'] + history2.history['mae']
all_val_mae = history1.history['val_mae'] + history2.history['val_mae']
phase1_epochs = len(history1.history['loss'])

epochs_range = range(1, len(all_loss) + 1)

axes[0].plot(epochs_range, all_loss, label='Train Loss')
axes[0].plot(epochs_range, all_val_loss, label='Val Loss')
axes[0].axvline(phase1_epochs, color='gray', linestyle=':', label='Fine-tune start')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, all_mae, label='Train MAE')
axes[1].plot(epochs_range, all_val_mae, label='Val MAE')
axes[1].axvline(phase1_epochs, color='gray', linestyle=':', label='Fine-tune start')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Mean Absolute Error')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Conversao para TFLite

Converter o modelo Keras para TFLite com quantizacao opcional.

In [ ]:
# Precisamos criar um modelo que denormaliza o output
# para que o TFLite retorne peso em kg diretamente

class DenormalizeLayer(keras.layers.Layer):
    def __init__(self, mean, std, **kwargs):
        super().__init__(**kwargs)
        self.mean = mean
        self.std = std

    def call(self, inputs):
        return inputs * self.std + self.mean

    def get_config(self):
        config = super().get_config()
        config.update({'mean': float(self.mean), 'std': float(self.std)})
        return config

# Modelo com denormalizacao embutida
export_model = keras.Sequential([
    model,
    DenormalizeLayer(weight_mean, weight_std)
])

# Testar
for images, _ in test_ds.take(1):
    pred_normalized = model.predict(images[:3])
    pred_kg = export_model.predict(images[:3])
    print("Output normalizado:", pred_normalized.flatten())
    print("Output em kg:", pred_kg.flatten())
    print("Denorm manual:", denormalize_weight(pred_normalized.flatten()))

In [ ]:
# Conversao TFLite - float32 (mxima precisao)
converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
tflite_model = converter.convert()

tflite_path = 'cattle_weight_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"Modelo TFLite salvo: {tflite_path}")
print(f"Tamanho: {os.path.getsize(tflite_path) / (1024*1024):.2f} MB")

# Versao quantizada (int8 - menor, um pouco menos preciso)
def representative_dataset():
    for images, _ in train_ds.take(50):
        for i in range(images.shape[0]):
            yield [tf.expand_dims(images[i], 0)]

converter_quant = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter_quant.optimizations = [tf.lite.Optimize.DEFAULT]
converter_quant.representative_dataset = representative_dataset
tflite_model_quant = converter_quant.convert()

tflite_quant_path = 'cattle_weight_model_quantized.tflite'
with open(tflite_quant_path, 'wb') as f:
    f.write(tflite_model_quant)

print(f"\nModelo quantizado salvo: {tflite_quant_path}")
print(f"Tamanho: {os.path.getsize(tflite_quant_path) / (1024*1024):.2f} MB")

## 10. Validar Modelo TFLite

In [ ]:
def test_tflite_model(tflite_path, test_dataset, label):
    """Testa modelo TFLite e compara com o Keras"""
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    print(f"\n=== {label} ===")
    print(f"Input shape: {input_details[0]['shape']}")
    print(f"Input dtype: {input_details[0]['dtype']}")
    print(f"Output shape: {output_details[0]['shape']}")
    print(f"Output dtype: {output_details[0]['dtype']}")
    
    predictions = []
    actuals = []
    
    for images, labels in test_dataset:
        for i in range(images.shape[0]):
            input_data = tf.expand_dims(images[i], 0)
            
            # Quantize input if needed
            if input_details[0]['dtype'] == np.uint8:
                input_scale, input_zero_point = input_details[0]['quantization']
                input_data = input_data / input_scale + input_zero_point
                input_data = tf.cast(input_data, tf.uint8)
            
            interpreter.set_tensor(input_details[0]['index'], input_data.numpy())
            interpreter.invoke()
            output = interpreter.get_tensor(output_details[0]['index'])
            
            # Dequantize output if needed
            if output_details[0]['dtype'] == np.uint8:
                output_scale, output_zero_point = output_details[0]['quantization']
                output = (output.astype(np.float32) - output_zero_point) * output_scale
            
            predictions.append(output.flatten()[0])
            actuals.append(denormalize_weight(labels[i].numpy()))
    
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    
    mae = np.mean(np.abs(predictions - actuals))
    rmse = np.sqrt(np.mean((predictions - actuals) ** 2))
    mape = np.mean(np.abs(predictions - actuals) / actuals) * 100
    
    print(f"MAE: {mae:.1f} kg")
    print(f"RMSE: {rmse:.1f} kg")
    print(f"MAPE: {mape:.1f}%")
    
    return predictions, actuals

pred_f32, actual_f32 = test_tflite_model(tflite_path, test_ds, "Float32")
pred_q8, actual_q8 = test_tflite_model(tflite_quant_path, test_ds, "Quantizado (int8)")

In [ ]:
# Comparar precisao float32 vs quantizado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(actual_f32, pred_f32, alpha=0.6, s=20, color='#3366FF', label='Float32')
axes[0].plot([min(actual_f32), max(actual_f32)], [min(actual_f32), max(actual_f32)], 'r--')
axes[0].set_xlabel('Peso Real (kg)')
axes[0].set_ylabel('Peso TFLite (kg)')
axes[0].set_title('TFLite Float32')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(actual_q8, pred_q8, alpha=0.6, s=20, color='#2E7D32', label='Quantizado')
axes[1].plot([min(actual_q8), max(actual_q8)], [min(actual_q8), max(actual_q8)], 'r--')
axes[1].set_xlabel('Peso Real (kg)')
axes[1].set_ylabel('Peso TFLite (kg)')
axes[1].set_title('TFLite Quantizado (int8)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Download do Modelo

Baixe o arquivo `.tflite` e coloque em `assets/models/` no projeto do app.

**Instrucoes:**
1. Copie `cattle_weight_model.tflite` para `assets/models/cattle_weight_model.tflite`
2. Rode `npx expo prebuild` ou faca um novo build com `eas build`
3. No app, o toggle "Modo Offline" estara disponivel

In [ ]:
# Salvar metadados do treinamento
metadata = {
    'model_architecture': 'MobileNetV3-Small + custom head',
    'input_shape': [1, IMG_SIZE, IMG_SIZE, 3],
    'input_dtype': 'float32',
    'input_range': [0.0, 1.0],
    'output': 'peso em kg (float32)',
    'weight_mean_train': float(weight_mean),
    'weight_std_train': float(weight_std),
    'train_samples': len(X_train),
    'val_samples': len(X_val),
    'test_samples': len(X_test),
    'test_mae_kg': float(np.mean(np.abs(pred_f32 - actual_f32))),
    'test_rmse_kg': float(np.sqrt(np.mean((pred_f32 - actual_f32) ** 2))),
    'tflite_size_mb': os.path.getsize(tflite_path) / (1024*1024),
    'tflite_quant_size_mb': os.path.getsize(tflite_quant_path) / (1024*1024),
}

with open('training_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

# Download no Colab
try:
    from google.colab import files
    files.download(tflite_path)
    files.download(tflite_quant_path)
    files.download('training_metadata.json')
    print("\nDownload iniciado!")
except ImportError:
    print(f"\nArquivos salvos em:")
    print(f"  - {os.path.abspath(tflite_path)}")
    print(f"  - {os.path.abspath(tflite_quant_path)}")
    print(f"  - {os.path.abspath('training_metadata.json')}")